# 06 · Quick start: from waveforms to a NonLinLoc-ready catalogue

A compact version of the workshop for a **1–2 hour tutorial**. In one notebook we

1. **download** the ARGOS waveforms and station metadata from the FDSN service,
2. **pick** P and S phases with a deep-learning picker (PhaseNet, VolPick weights),
3. **associate** picks into events and locate them (PyOcto, Aluto 1D model),
4. **save** the catalogue and a ready-to-run **NonLinLoc** project for relocation.

It uses the same configuration (`workshop.yml`), data folders and helper functions as the
full-day notebooks 00–05, which go deeper into each step (denoising, model comparison,
magnitudes, validation and statistics).

| Step | Time |
|---|---|
| Setup and download | 10 min |
| Phase picking | 15–25 min |
| Association and location | 15 min |
| Export for NonLinLoc | 10 min |

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/tesfahiwet-yemane/SeismoML/blob/main/notebooks/06_quickstart_catalogue.ipynb)

In [ ]:
# Running on Google Colab? This cell installs everything and gets the data (2-4 minutes).
# On your own computer it does nothing.
import os, subprocess, sys
if "google.colab" in sys.modules:
    repo = "/content/SeismoML"
    if not os.path.isdir(repo):
        subprocess.run(["git", "clone", "--depth", "1", "https://github.com/tesfahiwet-yemane/SeismoML.git", repo], check=True)
    subprocess.run([sys.executable, f"{repo}/scripts/colab_setup.py"] + "".split(), check=True)
    sys.path.insert(0, f"{repo}/src")
    os.chdir(f"{repo}/notebooks")

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from obspy import UTCDateTime

import seismoml as sml
from seismoml import association, catalogue, data, nonlinloc, picking, plotting

plotting.use_style()
cfg = sml.load_config()

HOURS = 24                     # length of data to process from 00:00 UTC; use e.g. 6 for a short class
MODEL = picking.ModelSpec("PhaseNet", "volpick")
THRESHOLD = 0.2                # probability threshold for P and S picks
OUT = cfg.outputs / "quickstart"
OUT.mkdir(parents=True, exist_ok=True)

t0, t1 = cfg.t0, min(cfg.t0 + HOURS * 3600, cfg.t1)
print(f"{cfg.network}, {len(cfg.stations)} stations, {t0} to {t1}")
print(f"Results go to {OUT}")

## 1 · Download

Station metadata (StationXML with instrument responses) and waveforms come from the
EarthScope/IRIS FDSN web service, as in notebook 00. Waveforms are stored in the **SDS
archive** (`data/sds`), one miniSEED file per channel per day. Stations already on disk are
not downloaded again.

In [ ]:
table = data.station_table(cfg.path("stations"))
stations = data.active_stations(table, cfg.day)
fig, _ = plotting.event_map(stations, lat=cfg.lat_range, lon=cfg.lon_range,
                            title=f"ARGOS stations, {cfg.day.date}")

In [ ]:
def on_disk(sta: str) -> bool:
    files = cfg.sds.glob(f"{t0.year}/{cfg.network}/{sta}/HH?.D/*.{t0.julday:03d}")
    return len(list(files)) >= 3


missing = [s for s in cfg.stations if not on_disk(s)]
if missing or not cfg.inventory_file.exists():
    client = data.fdsn_client(cfg.provider)
    inv = data.download_inventory(client, cfg.network, cfg.stations, t0, t1, cfg.channels)
    cfg.inventory_file.parent.mkdir(parents=True, exist_ok=True)
    inv.write(str(cfg.inventory_file), format="STATIONXML")
    for sta in missing:
        st = data.download_waveforms(client, cfg.network, sta, t0, t1, cfg.channels, cfg.location)
        data.write_sds(st, cfg.sds)
        print(f"{sta}: {len(st)} traces downloaded")
print("Stations on disk:", sum(on_disk(s) for s in cfg.stations), "of", len(cfg.stations))

## 2 · Phase picking

**PhaseNet** ([Zhu & Beroza, 2019](https://doi.org/10.1093/gji/ggy423)) returns, for every
sample, the probability of a P arrival, an S arrival or noise. We use the **VolPick** weights
([Zhong & Tan, 2024](https://doi.org/10.1029/2024GL108438)), trained on volcanic
earthquakes. Picks are the probability peaks above `THRESHOLD`.

Stations are processed one at a time to keep memory low.

In [ ]:
model = MODEL.load()
tables = []
for sta in cfg.stations:
    try:
        st = data.load_waveforms(cfg.sds, cfg.network, sta, t0, t1)
    except FileNotFoundError:
        print(f"{sta}: no data, skipped")
        continue
    df = picking.run_picker(model, st, threshold=THRESHOLD)
    print(f"{sta}: {(df.phase == 'P').sum():4d} P, {(df.phase == 'S').sum():4d} S picks")
    tables.append(df)
tables = [t for t in tables if len(t)] or tables[:1]
picks = pd.concat(tables, ignore_index=True).sort_values("time").reset_index(drop=True)
picking.save_picks(picks, OUT / f"picks_{MODEL.label}.csv")
print(f"\n{len(picks)} picks saved")

In [ ]:
fig = plotting.probability_histograms({MODEL.label: picks})

### Look at the picker output

The probability curves for the busiest minute of the day on one station: P (blue) and S (red)
peaks above the threshold become picks (dashed lines).

In [ ]:
demo = picks[picks["station"] == cfg.demo_station]
if len(demo):
    busiest = demo.set_index("time").resample("1min").size().idxmax()
    w0 = UTCDateTime(busiest.timestamp()) - 10
    st_w = data.load_waveforms(cfg.sds, cfg.network, cfg.demo_station, w0 - 30, w0 + 90)
    ann = model.annotate(st_w)
    in_win = demo[(demo["time"] >= pd.Timestamp(w0.datetime, tz="UTC")) &
                  (demo["time"] <= pd.Timestamp((w0 + 60).datetime, tz="UTC"))]
    fig = plotting.probability_panel(st_w.slice(w0, w0 + 60), ann.slice(w0, w0 + 60), in_win,
                                     title=f"{cfg.network}.{cfg.demo_station} · {MODEL.label}")

## 3 · Association and location

A pick only becomes part of an event if several stations have picks that one source, at one
place and time, can explain. **PyOcto** ([Münchmeyer, 2024](https://doi.org/10.26443/seismica.v3i1.1130))
searches space and time with an octree and returns preliminary hypocentres, using the 1D
velocity model of Wilks et al. (2017). The settings are those of notebook 03.

In [ ]:
tt_table = association.build_travel_time_table(cfg.velocity_model_file, cfg.datum_km, cfg.cache,
                                               xdist_km=60, zdist_km=45, delta_km=0.1)
vp0, vs0 = association.surface_velocities(cfg.velocity_model_file)
assoc = association.make_associator(
    cfg.lat_range, cfg.lon_range, cfg.depth_range, cfg.datum_km, tt_table, vp0, vs0,
    tolerance=1.0, cutoff_km=30.0, n_picks=6, n_p_picks=3, n_s_picks=2, n_p_and_s_picks=2)
station_df = association.stations_frame(stations, assoc, cfg.datum_km)
events, assigned = association.associate(picks, station_df, assoc, cfg.datum_km)
print(f"{len(events)} events; {len(assigned)} of {len(picks)} picks associated")
events.head()

In [ ]:
fig = plotting.map_and_sections(stations, {f"{MODEL.label} ({len(events)} events)": events},
                                cfg.lat_range, cfg.lon_range, cfg.depth_range,
                                title=f"Preliminary locations, {cfg.day.date}")

In [ ]:
if len(events):
    best = events.assign(n=events["n_P"] + events["n_S"]).sort_values("n", ascending=False).iloc[0]
    t_ev = UTCDateTime(best["time"].timestamp())
    st_ev = data.load_waveforms(cfg.sds, cfg.network, cfg.stations, t_ev - 5, t_ev + 20)
    st_ev.detrend("demean").filter("bandpass", freqmin=2, freqmax=25, zerophase=True)
    dist = {s: float(np.hypot(r.x - best.x, r.y - best.y))
            for s, r in zip(stations["station"], station_df.itertuples())}
    fig = plotting.record_section(st_ev, assigned[assigned["event_id"] == best["event_id"]], dist,
                                  t_ev - 2, t_ev + 12,
                                  title=f"Best-recorded event {best['time']:%H:%M:%S}, "
                                        f"{int(best['n_P'])} P + {int(best['n_S'])} S picks")

## 4 · Save the catalogue and export for NonLinLoc

PyOcto locations are preliminary (grid search with a fixed tolerance). For publication-quality
hypocentres, relocate with **NonLinLoc** ([Lomax et al., 2000](http://alomax.free.fr/nlloc/)),
which gives full probability density functions and uncertainties. We write:

* `events.csv` and `assigned_picks.csv`: the catalogue, in the same format as notebook 03;
* `nonlinloc/`: a complete NonLinLoc project. It has one `NLLOC_OBS` file per event, station
  coordinates (`GTSRCE`), the velocity model (`LAYER`), a control file `nlloc.in` and a script
  `run_nlloc.sh` that runs Vel2Grid, Grid2Time and NLLoc.

Each pick's uncertainty is taken from its probability: 0.03 s for probability 1, up to
0.2 s for probability 0.

In [ ]:
catalogue.write_table(events, OUT / "events.csv")
picking.save_picks(assigned, OUT / "assigned_picks.csv")

project = nonlinloc.write_project(
    OUT / "nonlinloc", events, assigned, stations, cfg.velocity_model_file, cfg.centre,
    max_depth_km=cfg.depth_range[1], top_km=-cfg.datum_km)
print(f"{project['n_obs']} observation files in {project['folder']}")
for p in sorted(project["folder"].iterdir()):
    print("  ", p.name)

In [ ]:
first = sorted((project["folder"] / "obs").glob("*.nonlinloc"))
if first:
    print(first[0].name)
    print(first[0].read_text()[:700])

### Run the relocation

Install NonLinLoc from [github.com/alomax/NonLinLoc](https://github.com/alomax/NonLinLoc)
(see its README; compiling takes a few minutes), then in a terminal:

```bash
cd outputs/quickstart/nonlinloc
bash run_nlloc.sh
```

Each event gets a `loc/*.hyp` file, which is the same format as the reference catalogue. To read the
results back into a table, use the converter from the reference catalogue:

```python
files = sorted((OUT / "nonlinloc" / "loc").glob("*.grid0.loc.hyp"))
files = [f for f in files if ".sum." not in f.name]
relocated, relocated_picks = catalogue.nlloc_to_tables(files)
```

Next steps: double-difference relocation (hypoDD, GrowClust), magnitudes (notebook 03),
validation against analyst picks (notebook 04) and statistics (notebook 05).

## Exercises

1. Change `MODEL` to `picking.ModelSpec("PhaseNet", "instance")`. How many events do you get?
2. Raise `THRESHOLD` to 0.5. Which events survive?
3. Open `nonlinloc/nlloc.in` and find the search grid (`LOCGRID`). Does it cover the depth
   range of your events?